# UdaPlay Project

## Part 01 - Offline RAG

In this part you build the **VectorDB** that powers UdaPlay's internal knowledge, using ChromaDB.

The data lives in the `games/` folder. Each JSON file is one game (on one platform) and becomes one document in the collection:

```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Developer": "Polyphony Digital",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks...",
  "YearOfRelease": 1997
}
```

The reusable pieces live in [`vector_store.py`](vector_store.py) (`VectorStoreManager`) and [`config.py`](config.py); this notebook walks through using them.

### Setup

In [1]:
# Only needed for the Udacity workspace: use pysqlite3 if it is installed
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [2]:
import json
import os
import sys

sys.path.insert(0, os.getcwd())  # make the project modules importable from the notebook

from config import Settings
from vector_store import GAMES_COLLECTION, MEMORY_COLLECTION, VectorStoreManager, make_embedding_function, game_to_document

### Environment

Copy `.env.example` to `.env` and fill in `OPENAI_API_KEY`, `TAVILY_API_KEY` and (for the Vocareum proxy) `OPENAI_BASE_URL`. `Settings.from_env()` loads `.env` (or `config.env`) and validates the keys.

In [3]:
settings = Settings.from_env()
print("Endpoint :", settings.openai_base_url)
print("Embedding:", settings.embedding_model)
print("Chroma   :", settings.chroma_path)

Endpoint : https://openai.vocareum.com/v1
Embedding: text-embedding-3-small
Chroma   : C:\Users\bhavya.s.gowda\Training\Building agents work\bsg-4bootcampagents\udaplay\chromadb


### VectorDB instance

A persistent Chroma client: the index survives kernel restarts, so games are embedded only once.

In [4]:
client = VectorStoreManager.persistent_client(settings.chroma_path)

### Collection

Embeddings come from OpenAI (`text-embedding-3-small` by default). **Use the same embedding function whenever you load the collection later** - Part 02 does this through the same helper.

In [5]:
embedding_fn = make_embedding_function(settings)
games = VectorStoreManager(client, GAMES_COLLECTION, embedding_fn)  # get-or-create, cosine distance
print(f"Collection '{games.name}' currently holds {games.count()} documents")

Collection 'udaplay' currently holds 0 documents


### Add documents

One document per JSON file. The embedded text combines name, platform, year, genre, developer, publisher and description, so questions like *"Who developed FIFA 21?"* match on any of them. Loading is an **upsert**, so re-running this cell is safe.

In [6]:
# 1) A raw game file, exactly as provided
raw_path = os.path.join(settings.games_dir, "001.json")
with open(raw_path, encoding="utf-8") as f:
    raw_game = json.load(f)
print(json.dumps(raw_game, indent=2, ensure_ascii=False))

# 2) The text that gets embedded for it (see game_to_document in vector_store.py)
print("\nEmbedded text:\n ", game_to_document(raw_game))

{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Developer": "Polyphony Digital",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}

Embedded text:
  Gran Turismo [PlayStation 1] (1997) - Racing. Developer: Polyphony Digital. Publisher: Sony Computer Entertainment. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.


In [7]:
# 3) Embed and store every game (upsert, so re-running is safe)
loaded = games.load_games(settings.games_dir)
print(f"Loaded {loaded} game records; collection now holds {games.count()} documents")

Loaded 34 game records; collection now holds 34 documents


### Semantic search

Try some questions. `distance` is cosine distance - lower is more similar.

In [8]:
def show(query, n=3):
    print(f"\nQ: {query}")
    for hit in games.query(query, n_results=n):
        m = hit["metadata"]
        print(f"  {hit['distance']:.3f}  {m['Name']} [{m['Platform']}] {m['YearOfRelease']} - {m.get('Developer') or m['Publisher']}")

for q in [
    "Who developed FIFA 21?",
    "When was God of War Ragnarok released?",
    "What platform was Pokemon Red launched on?",
    "first 3D Mario platformer",
    "fighting games from Warner Bros",
]:
    show(q)


Q: Who developed FIFA 21?


  0.311  FIFA 21 [PlayStation 4] 2020 - EA Vancouver and EA Romania
  0.317  FIFA 21 [PlayStation 5] 2020 - EA Vancouver and EA Romania
  0.622  Forza Horizon 5 [Xbox Series X/S] 2021 - Playground Games

Q: When was God of War Ragnarok released?


  0.290  God of War Ragnarök [PlayStation 4] 2022 - Santa Monica Studio
  0.301  God of War Ragnarök [PlayStation 5] 2022 - Santa Monica Studio
  0.638  Elden Ring [PlayStation 5] 2022 - FromSoftware

Q: What platform was Pokemon Red launched on?


  0.364  Pokémon Red [Game Boy] 1996 - Game Freak
  0.500  Pokémon Gold and Silver [Game Boy Color] 1999 - Game Freak
  0.638  Super Mario Bros. [NES] 1985 - Nintendo R&D4

Q: first 3D Mario platformer


  0.295  Super Mario 64 [Nintendo 64] 1996 - Nintendo EAD
  0.389  Super Mario Odyssey [Nintendo Switch] 2017 - Nintendo EPD
  0.451  Super Mario Bros. [NES] 1985 - Nintendo R&D4

Q: fighting games from Warner Bros


  0.569  Street Fighter II [Arcade] 1991 - Capcom
  0.571  Mortal Kombat X [PlayStation 4] 2015 - NetherRealm Studios
  0.595  Super Smash Bros. Ultimate [Nintendo Switch] 2018 - Bandai Namco Studios and Sora Ltd.


### Long-term memory collection

Part 02 also persists facts learned from the web. They go in a second collection in the same database, created here so both parts share one Chroma folder.

In [9]:
memory_store = VectorStoreManager(client, MEMORY_COLLECTION, embedding_fn)
print(f"Memory collection '{memory_store.name}' holds {memory_store.count()} remembered facts")

Memory collection 'udaplay_memory' holds 0 remembered facts


### The database is persistent

Open a brand-new client on the same folder: the embeddings are already there, so Part 02 (or a restarted kernel) does not need to re-embed anything.

In [10]:
reopened = VectorStoreManager(VectorStoreManager.persistent_client(settings.chroma_path), GAMES_COLLECTION, embedding_fn)
print(f"Reopened '{reopened.name}': {reopened.count()} documents")
print("Top hit for 'Zelda on Switch':", reopened.query("Zelda on Switch", 1)[0]["metadata"]["Name"])

Reopened 'udaplay': 34 documents


Top hit for 'Zelda on Switch': The Legend of Zelda: Breath of the Wild
